In [14]:
import pandas as pd

df = pd.read_csv("../data/synthetic_generator_data.csv")
df.head()

,Unnamed: 0,asset_id,site,issue,health_score,repairs
0,0,GEN000,Tampa,fuel contamination,1.99,moderate repair frequency
1,1,GEN001,New York,High Precipitates in Fuel,0.25,moderate repair frequency
2,2,GEN002,Buffalo,fuel contamination,1.45,high repair costs
3,3,GEN003,Buffalo,coolant leak,1.36,moderate repair frequency
4,4,GEN004,Buffalo,oil becoming worse quality,0.11,low repair frequency


In [15]:
def create_text(row):
    return (
        f"Generator {row['asset_id']} at {row['site']} has a health score of {row['health_score']}. "
        f"Observed issue: {row['issue']}. "
        f"Repair history: {row['repairs']}."
    )

df["text"] = df.apply(create_text, axis=1)


In [16]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = np.vstack(df["text"].apply(lambda x: model.encode(x)).values)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12432.40it/s]


In [17]:
query = "Which generators have oil issues?"
query_vec = model.encode([query])

D, I = index.search(query_vec, k=3)

results = df.iloc[I[0]]
results[["asset_id", "text"]]

,asset_id,text
158,GEN158,Generator GEN158 at LA has a health score of 1...
198,GEN198,Generator GEN198 at LA has a health score of 0...
174,GEN174,Generator GEN174 at Atlanta has a health score...


In [18]:
context = "\n".join(results["text"].tolist())

print(context)

Generator GEN158 at LA has a health score of 1.12. Observed issue: oil degradation. Repair history: high repair frequency.
Generator GEN198 at LA has a health score of 0.77. Observed issue: oil degradation. Repair history: moderate repair frequency.
Generator GEN174 at Atlanta has a health score of 0.2. Observed issue: oil degradation. Repair history: high repair costs.
